# Data Validation

This notebook shows a simple CSV validation flow with Frictionless and DigitalHub. It reads a CSV file, generates a report, and labels the dataset as valid or invalid.

## Setup

Initialize the environment and create a project.

In [ ]:
import os

import digitalhub as dh

p_name = f"tutorial-project-{os.environ['USER']}"
project = dh.get_or_create_project(p_name)

## Function Definition

Define the validation function and register it with the SDK.

In [ ]:
func = project.new_function(
    name="validate-csv",
    kind="python",
    python_version="PYTHON3_10",
    requirements="requirements.txt",
    code_src="src/functions.py",
    handler="main",
)

Test the function with a DataItem. For a quick run, register a local CSV file as the source.

In [ ]:
path_to_file = "./data-invalid.csv"
di = project.log_table(name="data-invalid.csv", source=path_to_file)

In [ ]:
func.run(action="build", wait=True)
run = func.run("job", inputs={"di": di.key}, wait=True)

The function runs as a batch job and produces a JSON report stored as an artifact. It also adds an `INVALID` label to the data item.

## Trigger

Set up a trigger to run the validation function when a CSV file is uploaded as a data item.

Create the trigger:

In [ ]:
func.trigger(
    "job",
    "lifecycle",
    "csv-trigger",
    states=["READY"],
    key=f"store://{p_name}/dataitem/table/*",
    template={"inputs": {"di": "{{input.key}}"}},
)

If you create a table data item in the console and upload a CSV file, the function runs when the item becomes ready.

You can also create a data item here:

In [ ]:
path_to_file = "./data-valid.csv"
project.log_table(name="data-valid.csv", source=path_to_file)